# 주간 차트 데이터 EDA
가설: "차트 포인트 합산 1위가 실제 '올해의 아티스트'나 '올해의 베스트송'을 받았는가?"  

분석: 연도별 차트 데이터 기반 예측 1위 vs 실제 수상자 매칭율 확인. 만약 다르다면 심사위원 점수나 투표가 어떤 변수가 되었는지 추론

### 심사대상
2024.10.31~2025.11.19 발매된 모든 음원  
각 부문별 음원 점수 집계  
단, 트랙제로 초이스 부문은 심사 기간 상이하고 핫트렌드 부문은 발매 시기 제약 없음

In [36]:
## 주간 차트 데이터 컬럼 정제
import pandas as pd

# 1. 2020~2025년 주간차트 데이터 불러오기
# 파일 리스트가 있다면 반복문으로 불러와서 한 번에 합치기
years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]
df_list = [pd.read_csv(f'/Users/js/MMA/Melon_Music_Award/data/주간차트/melon_{y}.csv') for y in years]

# 리스트를 통째로 전달
df_total = pd.concat(df_list, ignore_index=True)
df_total.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32600 entries, 0 to 32599
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   year        32600 non-null  int64 
 1   week        32600 non-null  object
 2   week_start  32600 non-null  object
 3   week_end    32600 non-null  object
 4   rank        32600 non-null  int64 
 5   title       32600 non-null  object
 6   artist      32600 non-null  object
 7   album       32600 non-null  object
 8   like_cnt    32600 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 2.2+ MB


In [199]:
df_total.head()

,year,week,week_start,week_end,rank,title,artist,album,like_cnt
0,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,1,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,495320
1,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,2,흔들리는 꽃들 속에서 네 샴푸향이 느껴진거야,장범준,멜로가 체질 OST Part 3,320118
2,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,3,조금 취했어 (Prod. 2soo),임재현,조금 취했어,136247
3,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,4,워커홀릭,볼빨간사춘기,Two Five,148140
4,2019,2019.09.30 ~ 2019.10.06,2019.09.30,2019.10.06,5,안녕,폴킴,호텔 델루나 OST Part.10,224259


In [200]:
# 중복 확인
df_total.duplicated().sum()

0

In [201]:
# 통계 확인
df_total['rank'].describe()

count    32600.000000
mean        50.500000
std         28.866513
min          1.000000
25%         25.750000
50%         50.500000
75%         75.250000
max        100.000000
Name: rank, dtype: float64

In [203]:
df_total.to_csv('/Users/js/MMA/Melon_Music_Award/data/melon_total.csv', index=False, encoding='utf-8-sig')

## 집계기간 별 데이터 나누기
###MMA 집계기간 정의  
2025 MMA 기준: 2024.10.31 ~ 2025.11.19  
→ 매년 전년도 10.31 ~ 당해년도 11.19 가정한다.

In [ ]:
import pandas as pd

# 1. 데이터 불러오기
df_total = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/주간차트/melon_total.csv')

# 2. 날짜 컬럼을 데이터타입으로 변환 (비교를 위해 필수)
df_total['week_start'] = pd.to_datetime(df_total['week_start'], format="mixed")
df_total['week_end'] = pd.to_datetime(df_total['week_end'], format='mixed')


In [ ]:
# 3. 각 MMA 연도별 집계 기간 설정(2020~2022 집계기간은 가장 최근 2025의 집계기간으로 대체)
mma_periods = {
    "MMA_2020": ('2019-10-31', '2020-11-19'),
    "MMA_2021": ('2020-10-31', '2021-11-19'),
    "MMA_2022": ('2021-10-31', '2022-11-19'),
    "MMA_2023": ('2022-11-04', '2023-11-01'),
    "MMA_2024": ('2023-11-02', '2024-10-30'),
    "MMA_2025": ('2024-10-31', '2025-11-19')
}

# 4. 기간별로 데이터 나누기
mma_datasets = {}

for mma_year, (start_date, end_date) in mma_periods.items():
    # 해당 기간에 포함되는 행만 필터링
    # 수정된 필터 조건: 종료일이 집계 시작일 이후이고, 시작일이 집계 종료일 이전인 경우
    mask = (df_total['week_end'] >= start_date) & (df_total['week_start'] <= end_date)
    mma_datasets[mma_year] = df_total.loc[mask].copy()
    
    # 잘 나눠졌는지 확인 출력
    print(f"{mma_year}: {len(mma_datasets[mma_year])}개의 행이 수집됨")


MMA_2020: 5600개의 행이 수집됨
MMA_2021: 5600개의 행이 수집됨
MMA_2022: 5600개의 행이 수집됨
MMA_2023: 5300개의 행이 수집됨
MMA_2024: 5300개의 행이 수집됨
MMA_2025: 5600개의 행이 수집됨


In [ ]:
# 5. 연도별 MMA 데이터만 따로 저장
for year, data in mma_datasets.items():
    data.to_csv(f'/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_{year}.csv', index=False)

## 주간차트 랭크로 성적 변환

주요 지표 선정  
- 기본점수 : 100~1점  

- 보너스점수 : 
MMA 특성을 고려해서 멜론차트 1위한 곡은 영향력을 다른 곡보다 크게 반영  
(확실히 1위곡은 영향력이 커야함 하위권 곡은 1~10위 진입이 없고 차트에만 들어간 느낌  
1위 한번 하는게 하위권 차트인 하는것보다 어려움)  
==> 1위곡 보너스 100점, 2위 30점, 3위 10점, 4~10위 5점

### 음원점수가 반영되는 수상부문
| 부문 | 설명 | 선정 방식 | 음원 심사 여부 |
|---|---|---|---|
| 카카오뱅크 올해의 앨범 | 최고 인기 앨범 | 음원 60% + 심사 20% + 투표 20% | O |
| 올해의 아티스트 | 최고 인기 아티스트 | 음원 60% + 심사 20% + 투표 20% | O |
| 올해의 베스트송 | 최고 인기 곡 | 음원 60% + 심사 20% + 투표 20% | O |
| 올해의 신인 | 최고 인기 신인 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 솔로 여자 | 최고 인기 솔로 여자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 솔로 남자 | 최고 인기 솔로 남자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 그룹 여자 | 최고 인기 그룹 여자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 그룹 남자 | 최고 인기 그룹 남자 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 OST | 최고 인기 드라마/영화/웹툰 등 작품 삽입곡 | 음원 60% + 심사 20% + 투표 20% | O |
| 베스트 팝 아티스트 | 최고 인기 POP 아티스트 | 음원 60% + 심사 20% + 투표 20% | O |
| TOP10 | 가장 사랑 받은 아티스트 10팀 | 음원 60% + 심사 20% + 투표 20%  | O |
| 밀리언스 TOP10 | 발매 24시간 내 100만 스트리밍 이상 달성하여 멜론의 전당에 오른 앨범 | 음원 80% + 투표 20%| O |
| 트랙제로 초이스 | 자신만의 장르와 스타일을 드러내 대중문화의 다양성에 기여한 곡과 아티스트 *심사기간 : 2024.10.05~2025.10.04 | 음원 20% + 심사 60% + 투표 20% | O |

### **최종점수 산출 방식 음원 70% + 좋아요 30%**
심사기준은 보통 음원 60% + 심사 20% + 투표 20%를 반영  
=> 투표는 각 곡의 '좋아요'로 점수를 대체 / 심사는 알 수 없으므로 음원점수, 투표점수에 10%씩 추가하여 최종 점수 집계  
(밀리언스 TOP10 => 음원 80% + 투표 20% / 트랙제로 초이스는 심사의 비중이 너무큼 )

### 집계기간별 데이터로 EDA

In [173]:
# 2025년 MMA
import pandas as pd

df_2025 = pd.read_csv("/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_2025.csv")
df_2025.head()
df_2024 = pd.read_csv("/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_2024.csv")


### 1위 100점, 2위 20점, 3위 10점, 4~10위 1점반영

In [190]:
import numpy as np
def calculate_mma_score(df_mma):
    # 1. 곡별로 그룹화 (title과 artist를 기준으로)
    # like_cnt는 주차별로 변할 수 있으므로 최대값을 기준으
    song_groups = df_mma.groupby(['title', 'artist'])
    
    # 2. 지표 계산
    # 기본 점수: 101 - rank (1위 100점 ~ 100위 1점)
    df_mma['base_score'] = 101 - df_mma['rank']
    
    # 보너스 점수 계산 함수
    # 차트 1위는 2~100위 보다 강력한 점수 반영
    def get_bonus(rank):
        if rank == 1: return 100
        elif rank == 2: return 20
        elif rank == 3: return 10
        elif rank <= 10: return 1
        else: return 0
    
    df_mma['bonus_score'] = df_mma['rank'].apply(get_bonus)
    
    # 3. 곡별 최종 집계
    mma_final = song_groups.agg(
        album=('album', 'first'),
        total_base_score=('base_score', 'sum'),
        total_bonus_score=('bonus_score', 'sum'),
        chart_in_weeks=('rank', 'count'),      # 차트인 기간 (주차 수)
        top1_count=('rank', lambda x: (x == 1).sum()),  # Top 1 횟수
        top10_count=('rank', lambda x: (x <= 10).sum()), # Top 10 횟수
        max_likes=('like_cnt', 'max')         # 좋아요 수 (팬덤 지표)
    ).reset_index()
    
    # 4. 좋아요 수를 정규화
    mma_final['like_score'] = np.sqrt(mma_final['max_likes'])
    mma_final['like_score'] = (mma_final['like_score'] / mma_final['like_score'].max()) * 100
    
    # 5. 종합 점수 계산 (가중치는 필요에 따라 조정)
    # 5-1. 음원 점수(기본+보너스) 합산
    mma_final['raw_music_score'] = mma_final['total_base_score'] + mma_final['total_bonus_score']

    # # 5-2. 음원 점수를 0~100점으로 정규화 (1위 곡이 100점이 되도록)
    max_music = mma_final['raw_music_score'].max()
    mma_final['normalized_music_score'] = (mma_final['raw_music_score'] / max_music) * 100

    # 5-3. 최종 점수 산출 (음원 70% + 좋아요 30%)
    mma_final['total_score'] = (
        (mma_final['normalized_music_score'] * 0.7) + 
        (mma_final['like_score'] * 0.3)
    )

    # 순위 정렬
    mma_final = mma_final.sort_values(by='total_score', ascending=False)
    
    return mma_final

# 사용 예시 (2025 MMA 데이터 적용)
mma_2024_scored = calculate_mma_score(df_2024)


In [191]:
mma_2024_scored.head(20)

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
265,헤어지자 말해요,박재정,1집 Alone,4287,16,53,0,16,193508,61.610006,4303,98.557032,87.472924
52,Hype Boy,NewJeans,NewJeans 1st EP 'New Jeans',3960,0,53,0,0,315188,78.629701,3960,90.700870,87.079520
53,I AM,IVE (아이브),I've IVE,4133,1,53,0,1,238147,68.347769,4134,94.686212,86.784679
145,그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection)),너드커넥션 (Nerd Connection),그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection)),4208,11,53,0,11,171102,57.933442,4219,96.633074,85.023184
161,너의 모든 순간,성시경,별에서 온 그대 OST Part.7,3746,0,53,0,0,322134,79.491386,3746,85.799359,83.906967
199,비의 랩소디,임재현,비의 랩소디,4025,341,48,3,17,109481,46.341615,4366,100.000000,83.902485
69,Love wins all,아이유,The Winning,3545,448,41,4,15,222390,66.047961,3993,91.456711,83.834086
92,Seven (feat. Latto) - Clean Ver.,정국,Seven (feat. Latto) - Clean Ver.,3946,9,53,0,9,230704,67.271227,3955,90.586349,83.591813
227,예뻤어,DAY6 (데이식스),Every DAY6 February,3588,4,48,0,4,380427,86.384824,3592,82.272103,83.505919
262,한 페이지가 될 수 있게,DAY6 (데이식스),The Book of Us : Gravity,3594,17,45,0,17,367301,84.881461,3611,82.707284,83.359537


In [193]:
years = ['2020', '2021', '2022', '2023', '2024', '2025']

mma_results = {}

# 2019~2025MMA 데이터 별 점수 추가 
for year in years:
    df = pd.read_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_{year}.csv")
    scored_df = calculate_mma_score(df)
    mma_results[year] = scored_df

# 추가된 점수 저장
for year, df in mma_results.items():
    df.to_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_{year}_v1.csv", index=False)

### 기존 보너스 점수 계산 + 1위횟수한 만큼 추가 점수 반영

In [197]:
import numpy as np
def calculate_mma_score(df_mma):
    # 1. 곡별로 그룹화 (title과 artist를 기준으로)
    # like_cnt는 주차별로 변할 수 있으므로 최대값을 기준으
    song_groups = df_mma.groupby(['title', 'artist'])
    
    # 2. 지표 계산
    # 기본 점수: 101 - rank (1위 100점 ~ 100위 1점)
    df_mma['base_score'] = 101 - df_mma['rank']
    
    # 보너스 점수 계산 함수
    # 차트 1위는 2~100위 보다 강력한 점수 반영
    def get_bonus(rank):
        if rank == 1: return 100
        elif rank == 2: return 20
        elif rank == 3: return 10
        elif rank <= 10: return 1
        else: return 0
    
    df_mma['bonus_score'] = df_mma['rank'].apply(get_bonus)
    
    # 3. 곡별 최종 집계
    mma_final = song_groups.agg(
        album=('album', 'first'),
        total_base_score=('base_score', 'sum'),
        total_bonus_score=('bonus_score', 'sum'),
        chart_in_weeks=('rank', 'count'),      # 차트인 기간 (주차 수)
        top1_count=('rank', lambda x: (x == 1).sum()),  # Top 1 횟수
        top10_count=('rank', lambda x: (x <= 10).sum()), # Top 10 횟수
        max_likes=('like_cnt', 'max')         # 좋아요 수 (팬덤 지표)
    ).reset_index()
    
    # 4. 좋아요 수를 정규화
    mma_final['like_score'] = np.sqrt(mma_final['max_likes'])
    mma_final['like_score'] = (mma_final['like_score'] / mma_final['like_score'].max()) * 100
    
    # 5. 종합 점수 계산 (가중치는 필요에 따라 조정)
    # 5-1. 음원 점수(기본+보너스) 합산
    mma_final['raw_music_score'] = mma_final['total_base_score'] + mma_final['total_bonus_score']

    # # 5-2. 음원 점수를 0~100점으로 정규화 (1위 곡이 100점이 되도록)
    max_music = mma_final['raw_music_score'].max()
    mma_final['normalized_music_score'] = (mma_final['raw_music_score'] / max_music) * 100

    # 5-3. 최종 점수 산출 (음원 70% + 좋아요 30%)
    mma_final['total_score'] = (
        (mma_final['normalized_music_score'] * 0.7) + 
        (mma_final['like_score'] * 0.3)
    )
    # 5-4. 1위 횟수 만큼 추가 점수 반영
    mma_final['total_score'] += mma_final['top1_count'] * 1
    
    # 순위 정렬
    mma_final = mma_final.sort_values(by='total_score', ascending=False)
    
    return mma_final

# 사용 예시 (2025 MMA 데이터 적용)
mma_2025_scored = calculate_mma_score(df_2025)


In [198]:
mma_2025_scored.head(20)

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
46,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618
29,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704
5,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116
120,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963
207,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915
247,한 페이지가 될 수 있게,DAY6 (데이식스),The Book of Us : Gravity,4450,0,56,0,0,367301,84.881377,4450,72.974746,76.546735
43,HAPPY,DAY6 (데이식스),Fourever,4872,17,56,0,17,213258,64.677620,4889,80.173827,75.524965
138,나는 반딧불,황가람,나는 반딧불,4982,59,56,0,32,158266,55.717956,5041,82.666448,74.581900
112,TOO BAD (feat. Anderson .Paak),G-DRAGON,Übermensch,3224,916,39,9,16,155640,55.253777,4140,67.891112,73.099911
213,예뻤어,DAY6 (데이식스),Every DAY6 February,4028,0,56,0,0,380427,86.384740,4028,66.054444,72.153533


In [199]:
years = ['2020', '2021', '2022', '2023', '2024', '2025']

mma_results = {}

# 2019~2025MMA 데이터 별 점수 추가 
for year in years:
    df = pd.read_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_{year}.csv")
    scored_df = calculate_mma_score(df)
    mma_results[year] = scored_df

# 추가된 점수 저장
for year, df in mma_results.items():
    df.to_csv(f"/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_{year}_v2.csv", index=False)

# 적용한 점수를 토대로 곡,앨범,아티스트 점수 구하기

In [23]:
import pandas as pd

df_2025 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA 집계기간별 데이터/MMA_scored_2025_v2.csv')
df_2025.head()

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618
1,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704
2,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116
3,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915


In [25]:
artist_df = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/artist_info_final.csv')
artist_df['artist'] = artist_df['아티스트명']
artist_df.head()

,아티스트명,국적,활동장르,데뷔,멤버,성별,구성,artist
0,로제 (ROSÉ),대한민국,"록/메탈, POP, 발라드",2021.03.12,NaN,여성,솔로,로제 (ROSÉ)
1,aespa,대한민국,"댄스, 일렉트로니카, R&B/Soul",2020.11.17,"카리나 (KARINA), 지젤 (GISELLE), 윈터 (WINTER), 닝닝 (N...",여성,그룹,aespa
2,제니 (JENNIE),대한민국,"댄스, 국외드라마, POP",2018.11.12,NaN,여성,솔로,제니 (JENNIE)
3,DAY6 (데이식스),대한민국,"록/메탈, 댄스, 발라드",2015.09.07,"성진 (DAY6), Young K (DAY6), 원필 (DAY6), 도운 (DAY6)",남성,그룹,DAY6 (데이식스)
4,G-DRAGON,대한민국,"댄스, 랩/힙합, 발라드",2009.08.18,NaN,남성,솔로,G-DRAGON


In [27]:
# 두 데이터 합치기
merged_2025 = df_2025.merge(
    artist_df,
    on='artist',
    how='left'
)

In [28]:
merged_2025.head()

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,아티스트명,국적,활동장르,데뷔,멤버,성별,구성
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618,G-DRAGON,대한민국,"댄스, 랩/힙합, 발라드",2009.08.18,NaN,남성,솔로
1,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704,WOODZ,대한민국,"댄스, R&B/Soul, 록/메탈",2016.07.29,NaN,남성,솔로
2,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116,로제 (ROSÉ),대한민국,"록/메탈, POP, 발라드",2021.03.12,NaN,여성,솔로
3,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963,aespa,대한민국,"댄스, 일렉트로니카, R&B/Soul",2020.11.17,"카리나 (KARINA), 지젤 (GISELLE), 윈터 (WINTER), 닝닝 (N...",여성,그룹
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915,AKMU (악뮤),대한민국,"댄스, 포크/블루스, 발라드",2014.04.07,"이찬혁, 이수현",혼성,그룹


In [60]:
merged_2025 = merged_2025.drop(columns=['아티스트명','국적','활동장르','멤버'])
merged_2025.head()

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score,데뷔,성별,구성
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.000000,101.123618,2009.08.18,남성,솔로
1,Drowning,WOODZ,OO-LI,5227,571,56,2,46,295867,76.181519,5798,95.080354,91.410704,2016.07.29,남성,솔로
2,APT.,로제 (ROSÉ),APT.,4935,603,56,4,27,220555,65.774842,5538,90.816661,87.304116,2021.03.12,여성,솔로
3,Whiplash,aespa,Whiplash - The 5th Mini Album,5200,218,56,0,42,152272,54.652671,5418,88.848803,78.589963,2020.11.17,여성,그룹
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,4170,1,56,0,1,495297,98.567607,4171,68.399475,77.449915,2014.04.07,혼성,그룹


In [65]:
albums_2025 = merged_2025['album'].unique().tolist()
albums_2025

['HOME SWEET HOME (feat. 태양, 대성)',
 'OO-LI',
 'APT.',
 'Whiplash - The 5th Mini Album',
 '항해',
 'The Book of Us : Gravity',
 'Fourever',
 '나는 반딧불',
 'Übermensch',
 'Every DAY6 February',
 '선재 업고 튀어 OST Part 1',
 "천상연 (웹툰 '선녀외전' X 이창섭 (LEE CHANGSUB))",
 '너에게 닿기를',
 '내게 사랑이 뭐냐고 물어본다면',
 'Armageddon - The 1st Album',
 '전설',
 '별에서 온 그대 OST Part.7',
 'rosie',
 'IVE EMPATHY',
 '오늘만 I LOVE YOU',
 'The Winning',
 "'키스 먼저 할까요?' OST Part.3",
 '그대만 있다면 (여름날 우리 X 너드커넥션 (Nerd Connection))',
 'PO￦ER',
 '모르시나요',
 '만화 (滿花)',
 'TWS 1st Mini Album ‘Sparkling Blue’',
 'Ruby',
 "1st Mini Album 'MANITO'",
 "2nd Mini Album 'Algorithm's Blossom'",
 '뛰어(JUMP)',
 'SYNK : PARALLEL LINE - Special Digital Single',
 '슬픈 초대장',
 'FAMOUS',
 'How Sweet',
 "NewJeans 1st EP 'New Jeans'",
 "I've IVE",
 '다정히 내 이름을 부르면 (경서예지 x 전건호)',
 'SUPER REAL ME',
 '사랑인가 봐 (사내맞선 OST 스페셜 트랙)',
 '2',
 '신사와 아가씨 OST Part.2',
 '에피소드',
 '1집 Alone',
 '비의 랩소디',
 'DRIP',
 'Supersonic',
 '어제보다 슬픈 오늘',
 '청혼하지 않을 이유를 못 찾았어',
 '인사',
 '16 Fantasy',


### 곡 점수

In [29]:
# 2025 MMA 기간 음원별 점수
song_2025 = merged_2025.copy()
song_2025[['title','artist','total_score']].head()

,title,artist,total_score
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618
1,Drowning,WOODZ,91.410704
2,APT.,로제 (ROSÉ),87.304116
3,Whiplash,aespa,78.589963
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),77.449915


### 아티스트 점수

In [30]:
artist_2025 = song_2025.groupby('artist').agg(
    artist_score=('total_score', 'sum'),
    song_count=('title', 'count')
)
artist_2025 = artist_2025.sort_values(by='artist_score', ascending=False).reset_index()

artist_2025.head()

,artist,artist_score,song_count
0,DAY6 (데이식스),410.163882,13
1,G-DRAGON,359.133845,10
2,aespa,322.881713,8
3,임영웅,287.308231,18
4,NewJeans,229.926558,7


### 앨범 점수

In [31]:
album_2025 = song_2025.groupby(['album', 'artist']).agg(
    album_score=('total_score', 'sum'),
    track_count=('title', 'count')
)
album_2025 = album_2025.sort_values(by='album_score', ascending=False).reset_index()

album_2025.head()

,album,artist,album_score,track_count
0,Fourever,DAY6 (데이식스),143.398214,2
1,Übermensch,G-DRAGON,141.663692,6
2,IVE EMPATHY,IVE (아이브),102.720940,2
3,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618,1
4,Armageddon - The 1st Album,aespa,97.903811,2


## 수상작 예측

In [9]:
# 올해의 베스트송
best_song = song_2025.sort_values('total_score', ascending=False).head(1)
best_song

,title,artist,album,total_base_score,total_bonus_score,chart_in_weeks,top1_count,top10_count,max_likes,like_score,raw_music_score,normalized_music_score,total_score
0,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",4896,1202,53,11,29,229386,67.078728,6098,100.0,101.123618


In [11]:
# 올해의 아티스트
artist_award = artist_2025.sort_values('artist_score', ascending=False).head(1)
artist_award

,artist,artist_score,song_count
0,DAY6 (데이식스),410.163882,13


In [12]:
# 올해의 앨범
album_award = album_2025.sort_values('album_score', ascending=False).head(1)
album_award

,album,artist,album_score,track_count
0,Fourever,DAY6 (데이식스),143.398214,2


In [32]:
# 올해의 신인(2명)
target_year = 2025

rookie = artist_df[
    artist_df['데뷔'].str[:4].astype(int) == target_year
]

rookie_award = artist_2025[
    artist_2025['artist'].isin(rookie['아티스트명'])
].sort_values('artist_score', ascending=False).head(2)

rookie_award

,artist,artist_score,song_count
17,ALLDAY PROJECT,92.328227,3
34,조째즈,59.236083,1


In [123]:
# 베스트 솔로(여자)
female_solo_award = (
    merged_2025[
        (merged_2025['성별'] == '여성') &
        (merged_2025['구성'] == '솔로')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score', ascending=False)
)

female_solo_award.head()

,artist,total_score
6,아이유,183.466531
3,로제 (ROSÉ),174.139933
11,제니 (JENNIE),111.220155
9,이영지,43.480807
14,태연 (TAEYEON),37.945810


In [118]:
# 베스트 솔로(여자)
male_solo_award = (
    merged_2025[
        (merged_2025['성별'] == '남성') &
        (merged_2025['구성'] == '솔로')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score', ascending=False)
)

male_solo_award.head()

,artist,total_score
4,G-DRAGON,359.133845
27,임영웅,287.308231
24,이무진,179.494979
38,황가람,105.911982
5,WOODZ,98.143903


In [119]:
# 베스트 그룹(여자)
female_group_award = (
    merged_2025[
        (merged_2025['성별'] == '여성') &
        (merged_2025['구성'] == '그룹')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score',ascending=False)
)

female_group_award.head()

,artist,total_score
16,aespa,322.881713
12,NewJeans,229.926558
6,IVE (아이브),226.807227
13,QWER,136.617193
20,아일릿(ILLIT),91.795234


In [122]:
# 베스트 그룹(남자)
male_group_award = (
    merged_2025[
        (merged_2025['성별'] == '남성') &
        (merged_2025['구성'] == '그룹')
    ]
    .groupby('artist')['total_score']
    .sum()
    .reset_index()
    .sort_values('total_score',ascending=False)
)

male_group_award.head()

,artist,total_score
3,DAY6 (데이식스),410.163882
8,PLAVE,177.589102
1,BOYNEXTDOOR,162.010572
9,RIIZE,108.045080
11,TWS (투어스),89.737420


In [121]:
# TOP10
top10 = artist_2025.sort_values('artist_score', ascending=False).head(10)
top10

,artist,artist_score,song_count
0,DAY6 (데이식스),410.163882,13
1,G-DRAGON,359.133845,10
2,aespa,322.881713,8
3,임영웅,287.308231,18
4,NewJeans,229.926558,7
5,IVE (아이브),226.807227,6
6,아이유,183.466531,9
7,이무진,179.494979,4
8,PLAVE,177.589102,13
9,로제 (ROSÉ),174.139933,3


In [57]:
# 밀리언스 TOP10
millions = album_2025.sort_values('album_score', ascending=False).head(10)
millions

,album,artist,album_score,track_count
0,Fourever,DAY6 (데이식스),143.398214,2
1,Übermensch,G-DRAGON,141.663692,6
2,IVE EMPATHY,IVE (아이브),102.720940,2
3,"HOME SWEET HOME (feat. 태양, 대성)",G-DRAGON,101.123618,1
4,Armageddon - The 1st Album,aespa,97.903811,2
5,OO-LI,WOODZ,91.410704,1
6,APT.,로제 (ROSÉ),87.304116,1
7,IM HERO,임영웅,86.064079,5
8,꽃갈피 셋,아이유,83.849268,6
9,FAMOUS,ALLDAY PROJECT,80.889286,2


In [113]:
rows = []

rows.append({
    '수상부문': '올해의 베스트송',
    '수상자': best_song.iloc[0]['title']
})

rows.append({
    '수상부문': '올해의 아티스트',
    '수상자': artist_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '올해의 앨범',
    '수상자': album_award.iloc[0]['album']
})

rows.append({
    '수상부문': '올해의 신인',
    '수상자': rookie_award.iloc[0]['artist']
})
rows.append({
    '수상부문': '올해의 신인',
    '수상자': rookie_award.iloc[1]['artist']
})


rows.append({
    '수상부문': '베스트 솔로 여자',
    '수상자': female_solo_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '베스트 솔로 남자',
    '수상자': male_solo_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '베스트 그룹 여자',
    '수상자': female_group_award.iloc[0]['artist']
})

rows.append({
    '수상부문': '베스트 그룹 남자',
    '수상자': male_group_award.iloc[0]['artist']
})

top10_list = top10['artist'].tolist()

rows.append({
    '수상부문': 'TOP10',
    '수상자': '/ '.join(top10_list)
})

millions_list = millions['album'].tolist()
rows.append({
    '수상부문': '밀리언스 TOP10',
    '수상자': '/ '.join(millions_list)
})

pre_award_2025 = pd.DataFrame(rows)

In [114]:
# 예측한 수상자 명단
pre_award_2025

,수상부문,수상자
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)"
1,올해의 아티스트,DAY6 (데이식스)
2,올해의 앨범,Fourever
3,올해의 신인,ALLDAY PROJECT
4,올해의 신인,조째즈
5,베스트 솔로 여자,아이유
6,베스트 솔로 남자,G-DRAGON
7,베스트 그룹 여자,aespa
8,베스트 그룹 남자,DAY6 (데이식스)
9,TOP10,DAY6 (데이식스)/ G-DRAGON/ aespa/ 임영웅/ NewJeans/ I...


In [95]:
pred_rookie = pre_award_2025[
    pre_award_2025['수상부문'] == '올해의 신인'
]['수상자'].tolist()

pred_rookie

['ALLDAY PROJECT', '조째즈']

In [115]:
pred_top10 = pre_award_2025[
    pre_award_2025['수상부문'] == 'TOP10'
]['수상자'].iloc[0]

pred_top10_list = [x.strip() for x in pred_top10.split('/')]
pred_top10_list

['DAY6 (데이식스)',
 'G-DRAGON',
 'aespa',
 '임영웅',
 'NewJeans',
 'IVE (아이브)',
 '아이유',
 '이무진',
 'PLAVE',
 '로제 (ROSÉ)']

In [116]:
pred_millions_top10 = pre_award_2025[
    pre_award_2025['수상부문'] == '밀리언스 TOP10'
]['수상자'].iloc[0]

pred_millions_list = [x.strip() for x in pred_millions_top10.split('/')]
pred_millions_list


['Fourever',
 'Übermensch',
 'IVE EMPATHY',
 'HOME SWEET HOME (feat. 태양, 대성)',
 'Armageddon - The 1st Album',
 'OO-LI',
 'APT.',
 'IM HERO',
 '꽃갈피 셋',
 'FAMOUS']

In [ ]:
# 실제 수상 명잔
real_award_2025 = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/MMA_수상내역/2025_MMA_수상내역.csv')
real_award_2025['수상부문'] = real_award_2025['category']


In [97]:
real_rookie = real_award_2025[
    real_award_2025['수상부문'] == '올해의 신인'
]['아티스트'].tolist()

real_rookie

['ALLDAY PROJECT', 'Hearts2Hearts (하츠투하츠)']

In [81]:
real_top10_list = real_award_2025[
    real_award_2025['수상부문'] == 'TOP10'
]['아티스트'].tolist()

real_top10_list

['로제 (ROSÉ)',
 '임영웅',
 '제니 (JENNIE)',
 'aespa',
 'BOYNEXTDOOR',
 'G-DRAGON',
 'IVE (아이브)',
 'NCT WISH',
 'PLAVE',
 'RIIZE']

In [101]:
real_millions_list = real_award_2025[
    real_award_2025['수상부문'] == '밀리언스 TOP10'
]['앨범'].tolist()

real_millions_list

['꽃갈피 셋',
 'Caligo Pt.1',
 'IM HERO 2',
 'IVE EMPATHY',
 'No Genre',
 'ODYSSEY - The 1st Album',
 'rosie',
 'Ruby',
 'SEVENTEEN 5th Album ‘HAPPY BURSTDAY’',
 'Übermensch']

In [130]:
exclude = ['올해의 신인','TOP10', '밀리언스 TOP10']

pred_single = pre_award_2025[
    ~pre_award_2025['수상부문'].isin(exclude)
]

real_single = real_award_2025[
    ~real_award_2025['수상부문'].isin(exclude)
]

compare_df = pred_single.merge(
    real_single,
    on='수상부문',
    how='inner',
    suffixes=('_pred', '_real')
)
compare_df

,수상부문,수상자,부문 구분,category,아티스트,곡명,앨범,이미지
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)",올해의 수상자,올해의 베스트송,G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",NaN,cdnimg.melon.co.kr/cm2/album/images/116/47/670...
1,올해의 아티스트,DAY6 (데이식스),올해의 수상자,올해의 아티스트,G-DRAGON,NaN,NaN,https://cdnimg.melon.co.kr/svc/images/mma/2025...
2,올해의 앨범,Fourever,올해의 수상자,올해의 앨범,G-DRAGON,NaN,Übermensch,https://cdnimg.melon.co.kr/cm2/album/images/11...
3,베스트 솔로 여자,아이유,베스트상,베스트 솔로 여자,로제 (ROSÉ),NaN,NaN,cdnimg.melon.co.kr/cm2/artistcrop/images/009/9...
4,베스트 솔로 남자,G-DRAGON,베스트상,베스트 솔로 남자,G-DRAGON,NaN,NaN,https://cdnimg.melon.co.kr/svc/images/mma/2025...
5,베스트 그룹 여자,aespa,베스트상,베스트 그룹 여자,IVE (아이브),NaN,NaN,cdnimg.melon.co.kr/cm2/artistcrop/images/030/5...
6,베스트 그룹 남자,DAY6 (데이식스),베스트상,베스트 그룹 남자,BOYNEXTDOOR,NaN,NaN,https://cdnimg.melon.co.kr/svc/images/mma/2025...


In [131]:
compare_df.drop(columns=['부문 구분', '이미지'])

,수상부문,수상자,category,아티스트,곡명,앨범
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)",올해의 베스트송,G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",NaN
1,올해의 아티스트,DAY6 (데이식스),올해의 아티스트,G-DRAGON,NaN,NaN
2,올해의 앨범,Fourever,올해의 앨범,G-DRAGON,NaN,Übermensch
3,베스트 솔로 여자,아이유,베스트 솔로 여자,로제 (ROSÉ),NaN,NaN
4,베스트 솔로 남자,G-DRAGON,베스트 솔로 남자,G-DRAGON,NaN,NaN
5,베스트 그룹 여자,aespa,베스트 그룹 여자,IVE (아이브),NaN,NaN
6,베스트 그룹 남자,DAY6 (데이식스),베스트 그룹 남자,BOYNEXTDOOR,NaN,NaN


In [132]:
def get_real_target(row):
    if row['수상부문'] == '올해의 베스트송':
        return row['곡명']
    elif row['수상부문'] == '올해의 아티스트':
        return row['아티스트']
    elif row['수상부문'] == '올해의 앨범':
        return row['앨범']
    elif row['수상부문'] == '베스트 솔로 여자':
        return row['아티스트']
    elif row['수상부문'] == '베스트 솔로 남자':
        return row['아티스트']
    elif row['수상부문'] == '베스트 그룹 여자':
        return row['아티스트']
    elif row['수상부문'] == '베스트 그룹 남자':
        return row['아티스트']
    else:
        return None  # 신인, TOP10 등 제외

compare_df['real_target'] = compare_df.apply(get_real_target, axis=1)

In [135]:
compare_df

,수상부문,수상자,부문 구분,category,아티스트,곡명,앨범,이미지,real_target
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)",올해의 수상자,올해의 베스트송,G-DRAGON,"HOME SWEET HOME (feat. 태양, 대성)",NaN,cdnimg.melon.co.kr/cm2/album/images/116/47/670...,"HOME SWEET HOME (feat. 태양, 대성)"
1,올해의 아티스트,DAY6 (데이식스),올해의 수상자,올해의 아티스트,G-DRAGON,NaN,NaN,https://cdnimg.melon.co.kr/svc/images/mma/2025...,G-DRAGON
2,올해의 앨범,Fourever,올해의 수상자,올해의 앨범,G-DRAGON,NaN,Übermensch,https://cdnimg.melon.co.kr/cm2/album/images/11...,Übermensch
3,베스트 솔로 여자,아이유,베스트상,베스트 솔로 여자,로제 (ROSÉ),NaN,NaN,cdnimg.melon.co.kr/cm2/artistcrop/images/009/9...,로제 (ROSÉ)
4,베스트 솔로 남자,G-DRAGON,베스트상,베스트 솔로 남자,G-DRAGON,NaN,NaN,https://cdnimg.melon.co.kr/svc/images/mma/2025...,G-DRAGON
5,베스트 그룹 여자,aespa,베스트상,베스트 그룹 여자,IVE (아이브),NaN,NaN,cdnimg.melon.co.kr/cm2/artistcrop/images/030/5...,IVE (아이브)
6,베스트 그룹 남자,DAY6 (데이식스),베스트상,베스트 그룹 남자,BOYNEXTDOOR,NaN,NaN,https://cdnimg.melon.co.kr/svc/images/mma/2025...,BOYNEXTDOOR


In [136]:
compare_df['correct'] = (
    compare_df['수상자'] ==
    compare_df['real_target']
)

In [138]:
compare_df[['수상부문','수상자','real_target']]

,수상부문,수상자,real_target
0,올해의 베스트송,"HOME SWEET HOME (feat. 태양, 대성)","HOME SWEET HOME (feat. 태양, 대성)"
1,올해의 아티스트,DAY6 (데이식스),G-DRAGON
2,올해의 앨범,Fourever,Übermensch
3,베스트 솔로 여자,아이유,로제 (ROSÉ)
4,베스트 솔로 남자,G-DRAGON,G-DRAGON
5,베스트 그룹 여자,aespa,IVE (아이브)
6,베스트 그룹 남자,DAY6 (데이식스),BOYNEXTDOOR


In [ ]:
accuracy = compare_df['correct'].mean()
print(f"정확도: {accuracy*100:.1f}%")


정확도: 28.6%


### 7개 수상부문 중 2개 정답(올해의 베스트송, 아티스트, 앨범, 베스트 솔로(남/여), 베스트 그룹(남/여)

In [103]:
def evaluate_multi_award(pred_list, real_list):
    pred_set = set(pred_list)
    real_set = set(real_list)
    
    return {
        '맞춘개수': len(pred_set & real_set),
        '정답': pred_set & real_set,
        '놓친': real_set - pred_set,
        '틀린예측': pred_set - real_set
    }

In [108]:
# 3종류 비교
# pred_rookie, real_rookie
# pred_top10_list, real_top10_list
# pred_millions_list, real_millions_list

In [124]:
evaluate_multi_award(pred_rookie, real_rookie)

{'맞춘개수': 1,
 '정답': {'ALLDAY PROJECT'},
 '놓친': {'Hearts2Hearts (하츠투하츠)'},
 '틀린예측': {'조째즈'}}

In [125]:
evaluate_multi_award(pred_top10_list, real_top10_list)

{'맞춘개수': 6,
 '정답': {'G-DRAGON', 'IVE (아이브)', 'PLAVE', 'aespa', '로제 (ROSÉ)', '임영웅'},
 '놓친': {'BOYNEXTDOOR', 'NCT WISH', 'RIIZE', '제니 (JENNIE)'},
 '틀린예측': {'DAY6 (데이식스)', 'NewJeans', '아이유', '이무진'}}

In [126]:
evaluate_multi_award(pred_millions_list, real_millions_list)

{'맞춘개수': 3,
 '정답': {'IVE EMPATHY', 'Übermensch', '꽃갈피 셋'},
 '놓친': {'Caligo Pt.1',
  'IM HERO 2',
  'No Genre',
  'ODYSSEY - The 1st Album',
  'Ruby',
  'SEVENTEEN 5th Album ‘HAPPY BURSTDAY’',
  'rosie'},
 '틀린예측': {'APT.',
  'Armageddon - The 1st Album',
  'FAMOUS',
  'Fourever',
  'HOME SWEET HOME (feat. 태양, 대성)',
  'IM HERO',
  'OO-LI'}}

## 제대로 예측하지 못한 이유

In [139]:
a = pd.read_csv('/Users/js/MMA/Melon_Music_Award/data/주간차트/melon_total.csv')
len(a['album'].unique())

1251